In [39]:
%pip install dmba


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score

from dmba import classificationSummary 

In [41]:
np.random.seed(1)

# read the right worksheet from excel
data = pd.read_excel("ebayAuctions.xlsx", sheet_name="eBay Auctions")

# analyze size of observations with null values
print(data.shape)
data.info()

(1972, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1972 entries, 0 to 1971
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Category      1972 non-null   object 
 1   Currency      1972 non-null   object 
 2   sellerRating  1972 non-null   int64  
 3   Duration      1972 non-null   int64  
 4   endDay        1972 non-null   object 
 5   ClosePrice    1972 non-null   float64
 6   OpenPrice     1972 non-null   float64
 7   Competitive?  1972 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 123.4+ KB


In [42]:
# nonnull values = rows for all columns so we proceed with data processing
# we begin by creating dummy variables and removing original categorical columns as requested

dummy_columns = ["Category", "Currency", "endDay", "Duration"]
for column in dummy_columns:
    if data[column].dtype != "string":
        data[column] = data[column].astype(str)
    
data = pd.get_dummies(data, columns = dummy_columns, drop_first=True)

In [43]:
# we now evaluate the numerical columns and check for their distributions and outliers
# Since we are working on a decision tree, the tree will find logical splits and we can keep them as numbers
    # Though I did realize this late and made a rudimentary splitting based on histograms
# Let's ensure that they are numbers

numerical_columns = ["sellerRating", "OpenPrice", "ClosePrice"]
for column in numerical_columns:
    if data[column].dtype != "float64" and data[column].dtype != "int64":
        data[column] = pd.to_numeric(data[column], errors='coerce')

# We can actually drop the closeprice since we will not have this in the incoming data
# turns out we do this later in the asignment - will keep here for now, and give data different name
data_noclose = data.drop(columns=["ClosePrice"])

# Let's now set the target variable to be boolean and move it to the end

data["Competitive?"] = data["Competitive?"].astype(bool)
data["Competitive?"] = data.pop("Competitive?")

# Double check
# print(data.shape)
# data.info()

In [44]:
# Data processing complete, ready for modeling
# Creating splits

X = data.drop(columns=['Competitive?'])
y = data['Competitive?']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)

# print("Training set size:", X_train.shape)
# print("Test set size:", X_test.shape)

# Creating decision tree, min_samples_split = 50
fullClassTree = DecisionTreeClassifier(min_samples_split=50, random_state = 1)
fullClassTree.fit(X_train, y_train)

#evaluation train accuracy
print("Training Accuracy (WITH CLOSE PRICE):")
classificationSummary(y_train, fullClassTree.predict(X_train))
# (8:32 pm 24 Nov) 78% - open to suggestions on how to improve accuracy


Training Accuracy (WITH CLOSE PRICE):
Confusion Matrix (Accuracy 0.8817)

       Prediction
Actual   0   1
     0 497  51
     1  89 546


In [45]:
# visualizing the tree
print("Decision Tree Representation (WITH CLOSE PRICE):")
feature_names = list(X.columns)
text_representation = tree.export_text(fullClassTree, feature_names = feature_names)
print(text_representation)

# Let's now use data_noclose to compare accuracy

data_noclose["Competitive?"] = data_noclose["Competitive?"].astype(bool)
data_noclose["Competitive?"] = data_noclose.pop("Competitive?")

# Double check
# print(data_noclose.shape)
# data_noclose.info()

Decision Tree Representation (WITH CLOSE PRICE):
|--- OpenPrice <= 3.62
|   |--- ClosePrice <= 3.64
|   |   |--- OpenPrice <= 1.12
|   |   |   |--- ClosePrice <= 1.12
|   |   |   |   |--- class: False
|   |   |   |--- ClosePrice >  1.12
|   |   |   |   |--- class: True
|   |   |--- OpenPrice >  1.12
|   |   |   |--- Category_Electronics <= 0.50
|   |   |   |   |--- ClosePrice <= 2.46
|   |   |   |   |   |--- sellerRating <= 874.50
|   |   |   |   |   |   |--- class: False
|   |   |   |   |   |--- sellerRating >  874.50
|   |   |   |   |   |   |--- sellerRating <= 914.50
|   |   |   |   |   |   |   |--- class: True
|   |   |   |   |   |   |--- sellerRating >  914.50
|   |   |   |   |   |   |   |--- OpenPrice <= 2.12
|   |   |   |   |   |   |   |   |--- class: False
|   |   |   |   |   |   |   |--- OpenPrice >  2.12
|   |   |   |   |   |   |   |   |--- class: False
|   |   |   |   |--- ClosePrice >  2.46
|   |   |   |   |   |--- OpenPrice <= 2.45
|   |   |   |   |   |   |--- class: True


In [46]:
# Data processing complete, ready for modeling
# Creating splits

X_noclose = data_noclose.drop(columns=['Competitive?'])
y_noclose = data_noclose['Competitive?']
X_train, X_test, y_train, y_test = train_test_split(X_noclose, y_noclose, test_size=0.4, random_state=1)

# print("Training set size:", X_train.shape)
# print("Test set size:", X_test.shape)

# Creating decision tree, min_samples_split = 50
fullClassTree = DecisionTreeClassifier(min_samples_split=50, random_state = 1)
fullClassTree.fit(X_train, y_train)

#evaluation train accuracy
print("Training Accuracy (WITHOUT CLOSE PRICE):")
classificationSummary(y_train, fullClassTree.predict(X_train))
# (8:32 pm 24 Nov) 78% - open to suggestions on how to improve accuracy


Training Accuracy (WITHOUT CLOSE PRICE):
Confusion Matrix (Accuracy 0.7794)

       Prediction
Actual   0   1
     0 404 144
     1 117 518


In [47]:
# visualizing the tree
print("Decision Tree Representation (WITHOUT CLOSE PRICE):")
feature_names = list(X_noclose.columns)
text_representation = tree.export_text(fullClassTree, feature_names = feature_names)
print(text_representation)

Decision Tree Representation (WITHOUT CLOSE PRICE):
|--- OpenPrice <= 3.62
|   |--- OpenPrice <= 1.12
|   |   |--- Duration_7 <= 0.50
|   |   |   |--- Category_Music/Movie/Game <= 0.50
|   |   |   |   |--- Category_Automotive <= 0.50
|   |   |   |   |   |--- Category_Collectibles <= 0.50
|   |   |   |   |   |   |--- class: True
|   |   |   |   |   |--- Category_Collectibles >  0.50
|   |   |   |   |   |   |--- class: True
|   |   |   |   |--- Category_Automotive >  0.50
|   |   |   |   |   |--- class: True
|   |   |   |--- Category_Music/Movie/Game >  0.50
|   |   |   |   |--- endDay_Sat <= 0.50
|   |   |   |   |   |--- OpenPrice <= 0.50
|   |   |   |   |   |   |--- class: True
|   |   |   |   |   |--- OpenPrice >  0.50
|   |   |   |   |   |   |--- class: True
|   |   |   |   |--- endDay_Sat >  0.50
|   |   |   |   |   |--- class: True
|   |   |--- Duration_7 >  0.50
|   |   |   |--- class: True
|   |--- OpenPrice >  1.12
|   |   |--- sellerRating <= 2483.00
|   |   |   |--- Category_B

In [48]:
keep_cols = ['ClosePrice', 'OpenPrice', 'sellerRating']
X = data[keep_cols]
y = data['Competitive?']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)

fullClassTree = DecisionTreeClassifier(min_samples_split=50, random_state = 1)
fullClassTree.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,50
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [49]:
# plot tree
feature_names = list(X.columns)
text_representation = tree.export_text(fullClassTree, feature_names = feature_names)
print(text_representation)

|--- OpenPrice <= 3.62
|   |--- ClosePrice <= 3.64
|   |   |--- OpenPrice <= 1.12
|   |   |   |--- ClosePrice <= 1.12
|   |   |   |   |--- class: False
|   |   |   |--- ClosePrice >  1.12
|   |   |   |   |--- class: True
|   |   |--- OpenPrice >  1.12
|   |   |   |--- OpenPrice <= 2.45
|   |   |   |   |--- ClosePrice <= 2.24
|   |   |   |   |   |--- sellerRating <= 987.00
|   |   |   |   |   |   |--- class: False
|   |   |   |   |   |--- sellerRating >  987.00
|   |   |   |   |   |   |--- class: False
|   |   |   |   |--- ClosePrice >  2.24
|   |   |   |   |   |--- class: True
|   |   |   |--- OpenPrice >  2.45
|   |   |   |   |--- sellerRating <= 21.00
|   |   |   |   |   |--- class: False
|   |   |   |   |--- sellerRating >  21.00
|   |   |   |   |   |--- ClosePrice <= 2.89
|   |   |   |   |   |   |--- class: False
|   |   |   |   |   |--- ClosePrice >  2.89
|   |   |   |   |   |   |--- class: False
|   |--- ClosePrice >  3.64
|   |   |--- OpenPrice <= 2.46
|   |   |   |--- class: Tr

In [50]:
print("Training Accuracy (Just close price, open price, seller rating):")
classificationSummary(y_train, fullClassTree.predict(X_train))

Training Accuracy (Just close price, open price, seller rating):
Confusion Matrix (Accuracy 0.8808)

       Prediction
Actual   0   1
     0 486  62
     1  79 556
